# Week 5: Foundation Fix Verification

This notebook verifies that the foundation gap has been addressed:
1. Image markers are extracted before stripping
2. Section headers are parsed and populated
3. Metadata is properly populated in chunks
4. Vector store works with new metadata

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PDF_DIR = Path('../data/raw_pdfs')
CHROMA_DIR = Path('../data/chroma_db_c3')

print(f"PDFs: {list(PDF_DIR.glob('*.pdf'))}")

## 1. Test Image Marker Extraction

In [ ]:
from src.parsing import extract_image_markers, extract_sections, parse_pdf

# Test on sample PDF
sample_pdf = list(PDF_DIR.glob('waterpurifier_complex.pdf'))[0]
parsed = parse_pdf(sample_pdf)

print(f"Source: {parsed.source}")
print(f"Category: {parsed.category}")
print(f"Complexity: {parsed.complexity}")
print(f"\nSections found ({len(parsed.sections)}):")
for s in parsed.sections[:10]:
    print(f"  - {s}")
if len(parsed.sections) > 10:
    print(f"  ... and {len(parsed.sections) - 10} more")

In [ ]:
# Check image markers
page = parsed.pages[0]
print(f"Image markers found: {len(page.image_markers)}")
print(f"\nFirst 5 image markers:")
for img in page.image_markers[:5]:
    print(f"  Position {img['position']}: {img['width']}x{img['height']}")

## 2. Test Chunking with Metadata

In [ ]:
from src.chunking import chunk_pdf, chunks_to_langchain_docs

# Chunk the sample PDF
chunks = chunk_pdf(sample_pdf, chunk_size=1000, chunk_overlap=200)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunks with sections: {sum(1 for c in chunks if c.section)}")
print(f"Chunks with images: {sum(1 for c in chunks if c.image_count > 0)}")
print(f"Total images across chunks: {sum(c.image_count for c in chunks)}")

In [ ]:
# Show sample chunks with metadata
print("Sample chunks with metadata:")
print("=" * 70)

for chunk in chunks[:5]:
    print(f"\n[{chunk.chunk_id}]")
    print(f"  Section: {chunk.section}")
    print(f"  Images: {chunk.image_count}")
    print(f"  Text preview: {chunk.text[:100]}...")

In [ ]:
# Find chunks with images and show their sections
print("Chunks with images:")
print("=" * 70)

image_chunks = [c for c in chunks if c.image_count > 0]
for chunk in image_chunks[:10]:
    print(f"\n[{chunk.chunk_id}] Section: {chunk.section}")
    print(f"  Images: {chunk.image_count}")
    for img in chunk.image_markers:
        print(f"    - {img['width']}x{img['height']} at pos {img['position']}")

## 3. Test LangChain Document Conversion

In [ ]:
# Convert to LangChain docs
docs = chunks_to_langchain_docs(chunks)

print(f"Total documents: {len(docs)}")
print(f"\nSample document metadata:")
print(docs[5].metadata)

In [ ]:
# Verify all metadata fields are populated
sample_meta = docs[5].metadata

required_fields = ['chunk_id', 'source', 'category', 'complexity', 
                   'page', 'section', 'chunk_index', 'char_count', 
                   'image_count', 'image_ids']

print("Metadata field check:")
for field in required_fields:
    value = sample_meta.get(field, 'MISSING')
    status = '✓' if field in sample_meta else '✗'
    print(f"  {status} {field}: {value}")

## 4. Create Vector Store with All PDFs

In [ ]:
from src.vectorstore import create_vectorstore

# Create vector store with all PDFs
vectorstore, all_chunks = create_vectorstore(
    pdf_dir=PDF_DIR,
    persist_dir=CHROMA_DIR,
    collection_name='lg_manuals_c3',
    chunk_size=1000,
    chunk_overlap=200,
)

In [ ]:
# Verify metadata in vector store
results = vectorstore._collection.get(limit=5, include=['metadatas'])

print("Sample metadata from vector store:")
for meta in results['metadatas']:
    print(f"\n{meta['chunk_id']}:")
    print(f"  category: {meta['category']}")
    print(f"  section: {meta.get('section', 'N/A')}")
    print(f"  image_count: {meta.get('image_count', 0)}")

## 5. Test Retrieval with New Metadata

In [ ]:
# Test queries
TEST_QUERIES = [
    {"query": "정수기 필터 교체는 어떻게 하나요?", "expected_category": "waterpurifier"},
    {"query": "공기청정기 필터 청소 방법 알려주세요", "expected_category": "airpurifier"},
    {"query": "청소기 배터리 충전 시간은 얼마나 되나요?", "expected_category": "vacuumcleaner"},
    {"query": "Wi-Fi 연결이 안될 때 어떻게 해야 하나요?", "expected_category": None},
]

retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

for q in TEST_QUERIES:
    docs = retriever.invoke(q['query'])
    print(f"\nQuery: {q['query'][:40]}...")
    print(f"Expected: {q['expected_category']}")
    print(f"Retrieved:")
    for i, doc in enumerate(docs):
        cat = doc.metadata.get('category', '?')
        section = doc.metadata.get('section', 'N/A')
        img_count = doc.metadata.get('image_count', 0)
        match = '✓' if q['expected_category'] is None or cat == q['expected_category'] else '✗'
        print(f"  {match} [{cat}] Section: {section[:30] if section else 'N/A'}... Images: {img_count}")

## 6. Section Distribution Analysis

In [ ]:
# Analyze section distribution
from collections import Counter

section_counts = Counter(c.section for c in all_chunks if c.section)

print(f"Total unique sections: {len(section_counts)}")
print(f"\nTop 15 sections by chunk count:")
for section, count in section_counts.most_common(15):
    print(f"  {count:3d} chunks: {section}")

In [ ]:
# Image distribution by section
section_images = {}
for chunk in all_chunks:
    if chunk.section:
        if chunk.section not in section_images:
            section_images[chunk.section] = 0
        section_images[chunk.section] += chunk.image_count

print("Sections with most images:")
sorted_sections = sorted(section_images.items(), key=lambda x: x[1], reverse=True)
for section, count in sorted_sections[:10]:
    print(f"  {count:3d} images: {section}")

## 7. Summary

**Foundation Gap Status:**
- [ ] Image markers extracted before stripping
- [ ] Section headers parsed from `##` patterns
- [ ] Metadata populated: section, image_count, image_ids
- [ ] Vector store created with new metadata

In [ ]:
# Final summary
chunks_with_section = sum(1 for c in all_chunks if c.section)
chunks_with_images = sum(1 for c in all_chunks if c.image_count > 0)
total_images = sum(c.image_count for c in all_chunks)

print("Foundation Gap Fix Summary")
print("=" * 50)
print(f"Total chunks: {len(all_chunks)}")
print(f"Chunks with section: {chunks_with_section} ({chunks_with_section/len(all_chunks)*100:.1f}%)")
print(f"Chunks with images: {chunks_with_images} ({chunks_with_images/len(all_chunks)*100:.1f}%)")
print(f"Total images tracked: {total_images}")
print(f"Unique sections: {len(section_counts)}")
print(f"\n✓ Foundation gap addressed!")